# rec_sentence 프로덕션 배치 생성 (오프라인 모드)

**목적**: 추천 풀 VOD × 5 세그먼트 → results.parquet 생성

**환경**: Google Colab A100 GPU (DB 직접 접속 불필요)

**모델**: `gemma3:12b-it-qat` 또는 `gemma3:27b-it-qat`

---

## 전체 흐름
```
로컬: export_for_colab.py → colab_data/vod_contexts.parquet
  ↓ Google Drive 업로드
Colab: batch_generate.py --offline → colab_data/results.parquet
  ↓ Google Drive 다운로드
로컬: ingest_results.py → serving.rec_sentence UPSERT
```

## 실행 순서
1. Ollama 설치 + 모델 다운로드
2. Google Drive 마운트 + 의존성 설치
3. 데이터 확인
4. 12B vs 27B 비교 (선택)
5. 배치 생성
6. 결과 확인

## 1. Ollama 설치 + Google Drive 마운트

In [ ]:
# Ollama 설치
!sudo apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

# Google Drive 마운트 (모델 캐시 + 데이터용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, time, os, shutil

# Ollama 서버 시작
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)
print(f"Ollama PID: {proc.pid}")

# ── 모델 캐시: Drive → 로컬 복원 ──
DRIVE_CACHE = "/content/drive/MyDrive/ollama_models"
LOCAL_MODELS = os.path.expanduser("~/.ollama/models")

if os.path.exists(DRIVE_CACHE) and os.listdir(DRIVE_CACHE):
    print("Drive 캐시 발견 → 로컬로 복원 중...")
    # 기존 로컬 모델 폴더 교체
    if os.path.exists(LOCAL_MODELS):
        shutil.rmtree(LOCAL_MODELS)
    shutil.copytree(DRIVE_CACHE, LOCAL_MODELS)
    print(f"복원 완료: {sum(f.stat().st_size for f in __import__('pathlib').Path(LOCAL_MODELS).rglob('*') if f.is_file()) / 1e9:.1f}GB")
    !ollama list
else:
    print("Drive 캐시 없음 → 다음 셀에서 모델 다운로드 필요")

In [ ]:
# ── 모델 다운로드 (캐시 복원 시 스킵) ──
# 위 셀에서 "Drive 캐시 발견"이 출력됐으면 이 셀은 실행 불필요

!ollama pull gemma3:12b-it-qat
!ollama pull gemma3:27b-it-qat

# ── 다운로드 후 Drive에 캐시 저장 (최초 1회) ──
import shutil, os

DRIVE_CACHE = "/content/drive/MyDrive/ollama_models"
LOCAL_MODELS = os.path.expanduser("~/.ollama/models")

if os.path.exists(LOCAL_MODELS):
    print("Drive에 모델 캐시 저장 중... (최초 1회, ~26GB, 수 분 소요)")
    if os.path.exists(DRIVE_CACHE):
        shutil.rmtree(DRIVE_CACHE)
    shutil.copytree(LOCAL_MODELS, DRIVE_CACHE)
    print(f"캐시 저장 완료: {DRIVE_CACHE}")
else:
    print("로컬 모델 없음 — pull 먼저 실행")

In [ ]:
!ollama list

## 2. 의존성 설치 + 데이터 경로 설정

### 사전 준비 (로컬에서 완료해야 할 것)
```bash
# 1. parquet 추출
python gen_rec_sentence/scripts/export_for_colab.py

# 2. Google Drive에 업로드할 폴더 구조:
#    내 드라이브/dxshcool/
#    ├── gen_rec_sentence/
#    │   ├── src/          (context_builder, sentence_generator, quality_filter)
#    │   ├── scripts/      (batch_generate.py)
#    │   └── data/
#    │       └── colab_data/
#    │           ├── vod_contexts.parquet
#    │           └── existing_pairs.parquet (있으면)
```

In [ ]:
import os

# ⚠️ Drive 폴더명 확인: dxshcool vs dxschool 등 본인 Drive 구조에 맞게 수정
PROJECT_ROOT = "/content/drive/MyDrive/dxschool"
OFFLINE_DIR = f"{PROJECT_ROOT}/gen_rec_sentence/data/colab_data"

assert os.path.exists(f"{OFFLINE_DIR}/vod_contexts.parquet"), \
    f"vod_contexts.parquet 없음! 로컬에서 export_for_colab.py 먼저 실행"
print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"오프라인 데이터: {OFFLINE_DIR}")

In [ ]:
!pip install -q python-dotenv ollama pandas pyarrow

In [ ]:
os.chdir(PROJECT_ROOT)
print(f"작업 디렉토리: {os.getcwd()}")

## 3. 데이터 확인

In [ ]:
import pandas as pd

df = pd.read_parquet(f"{OFFLINE_DIR}/vod_contexts.parquet")
print(f"VOD 컨텍스트: {len(df):,}건")
print(f"전체 생성 대상: {len(df) * 5:,}쌍 (VOD × 5 seg)")
print(f"\nct_cl 분포:")
print(df["ct_cl"].value_counts().to_string())
print(f"\n샘플 3건:")
for _, r in df.head(3).iterrows():
    print(f"  [{r['ct_cl']}] {r['asset_nm']}: {r['smry'][:60]}...")

# 기존 쌍 확인
exist_path = f"{OFFLINE_DIR}/existing_pairs.parquet"
if os.path.exists(exist_path):
    n_exist = len(pd.read_parquet(exist_path))
    print(f"\n기존 생성 쌍: {n_exist:,}건 → 실제 생성: {len(df) * 5 - n_exist:,}쌍")
else:
    print(f"\n기존 생성 쌍 없음 (첫 실행)")

## 4. 12B vs 27B 품질 비교 (선택)

동일 VOD 10건에 대해 두 모델을 비교. 27B가 확실히 좋으면 전체 배치를 27B로.

| 모델 | Q4 사이즈 | A100 VRAM 여유 | 전체 배치 예상 |
|------|----------|---------------|---------------|
| gemma3:12b-it-qat | ~8.9GB | 31GB | ~2-3h |
| gemma3:27b-it-qat | ~17GB | 23GB | ~5-7h |

In [ ]:
import sys, time
sys.path.insert(0, ".")
import ollama as _ollama
from gen_rec_sentence.src.sentence_generator import _parse_json_response

# 10건 샘플
sample = df.sample(n=min(10, len(df)), random_state=42)

MODELS = ["gemma3:12b-it-qat", "gemma3:27b-it-qat"]

_CMP_PROMPT = """\
당신은 IPTV VOD 콘텐츠 소개 작가입니다.
아래 VOD 정보를 바탕으로 홈 배너 포스터 하단에 표시할 소개 문구를 작성하세요.

규칙:
- 1~2문장, 20~80자 (공백 포함)
- 핵심 배경·상황·갈등을 구체적으로 — 시적·추상적 표현 금지
- 느낌표(!) 최대 1개 — 남발 금지
- 제목 반복 금지, 권유형/합쇼체 금지
- JSON: {{"rec_sentence": "..."}}

VOD 정보:
- 제목: {asset_nm}
- 장르: {genre_detail}
- 감독: {director}
- 출연: {cast_lead}
- 줄거리: {smry}

타겟 시청자: 액션·범죄·장르물을 즐기는 성인 시청자.
"""

results = {m: [] for m in MODELS}

for model in MODELS:
    print(f"\n{'='*60}")
    print(f"모델: {model}")
    print(f"{'='*60}")
    t0 = time.time()

    for _, r in sample.iterrows():
        prompt = _CMP_PROMPT.format(
            asset_nm=r["asset_nm"], genre_detail=r["genre_detail"],
            director=r["director"], cast_lead=r["cast_lead"],
            smry=str(r["smry"])[:300],
        )
        try:
            resp = _ollama.chat(model=model,
                               messages=[{"role": "user", "content": prompt}],
                               options={"temperature": 0.7})
            text = resp["message"]["content"].strip()
            sentence = _parse_json_response(text).get("rec_sentence", text)
        except Exception as e:
            sentence = f"[ERROR: {e}]"

        results[model].append({"nm": r["asset_nm"], "ct": r["ct_cl"], "sentence": sentence})
        print(f"  [{r['ct_cl']}] {r['asset_nm']}: {sentence}")

    elapsed = time.time() - t0
    print(f"\n  소요: {elapsed:.1f}초 ({elapsed/len(sample):.1f}초/건)")

In [ ]:
# 나란히 비교
print(f"\n{'='*80}")
print("12B vs 27B 나란히 비교")
print(f"{'='*80}")

m12, m27 = MODELS
for r12, r27 in zip(results[m12], results[m27]):
    print(f"\n[{r12['ct']}] {r12['nm']}")
    print(f"  12B: {r12['sentence']}")
    print(f"  27B: {r27['sentence']}")

for model in MODELS:
    lens = [len(r["sentence"]) for r in results[model] if not r["sentence"].startswith("[ERROR")]
    if lens:
        print(f"\n{model}: 평균 {sum(lens)/len(lens):.1f}자, 범위 {min(lens)}~{max(lens)}자")

print(f"\n⚠️  27B가 확실히 좋으면 아래 셀에서 --model gemma3:27b-it-qat 로 변경")
print(f"    비슷하면 12B 유지 (속도 2배 빠름)")

## 5. 배치 생성 — Ollama (순차, 느림)

⚠️ **Ollama는 순차 처리라 A100에서도 ~0.28건/초밖에 안 나옴.**
7만건이면 ~70시간 걸림. **섹션 5a (vLLM)를 대신 사용할 것을 권장.**

아래는 기존 Ollama 방식 (소규모 테스트 or 로컬 환경용).

## 5a. vLLM 백엔드 (권장 — 10~20배 빠름)

Ollama는 순차 처리라 A100 GPU 활용률이 낮음. **vLLM은 continuous batching으로 동시 16개 요청을 GPU가 한꺼번에 처리.**

| 백엔드 | 처리량 | 7만건 소요 |
|--------|--------|-----------|
| Ollama (순차) | ~0.28건/초 | ~70시간 |
| **vLLM (16병렬)** | **~4-5건/초** | **~4-5시간** |

**모델**: `hugging-quants/gemma-2-27b-it-AWQ-INT4` (AWQ 4bit, ~16GB VRAM → A100 40GB OK)

**vLLM 사용 시 Ollama 설치(섹션 1~4)는 불필요.** 이 섹션만 실행하면 됨.

In [ ]:
# 소규모 테스트 (10건)
!python gen_rec_sentence/scripts/batch_generate.py \
    --offline gen_rec_sentence/data/colab_data \
    --model gemma3:12b-it-qat \
    --limit 10 \
    --temperature 0.7

In [ ]:
# 전체 배치 (증분 — 이전 결과 자동 이어받기)
# 27B 선택 시: --model gemma3:27b-it-qat
!python gen_rec_sentence/scripts/batch_generate.py \
    --offline gen_rec_sentence/data/colab_data \
    --model gemma3:12b-it-qat \
    --temperature 0.7

## 6. 결과 확인

In [ ]:
import pandas as pd, os

result_path = f"{OFFLINE_DIR}/results.parquet"
if not os.path.exists(result_path):
    print("results.parquet 없음 — 배치 생성을 먼저 실행하세요")
else:
    df_r = pd.read_parquet(result_path)
    print(f"총 결과: {len(df_r):,}건")
    print(f"파일 크기: {os.path.getsize(result_path) / 1024 / 1024:.1f}MB")

    print(f"\n세그먼트별 통계:")
    for seg_id, grp in df_r.groupby("segment_id"):
        avg_len = grp["rec_sentence"].str.len().mean()
        print(f"  segment {seg_id}: {len(grp):>6,}건, 평균 {avg_len:.1f}자")

    print(f"\n랜덤 샘플 10건:")
    for i, (_, r) in enumerate(df_r.sample(n=min(10, len(df_r))).iterrows(), 1):
        print(f"  {i:>2}. [seg{r['segment_id']}] {r['rec_sentence']}")

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, ".")
from gen_rec_sentence.scripts.batch_generate import _build_prompt, _call_ollama
from gen_rec_sentence.src.quality_filter import validate

df = pd.read_parquet(f"{OFFLINE_DIR}/vod_contexts.parquet").head(10)
ctx_map = {row["vod_id"]: {col: str(row[col]) for col in df.columns} for _, row in df.iterrows()}

result_df = pd.read_parquet(f"{OFFLINE_DIR}/results.parquet")
existing = {(r["vod_id"], int(r["segment_id"])) for _, r in result_df.iterrows()}

fails = []
for vod_id, ctx in ctx_map.items():
    for seg_id in range(5):
        if (vod_id, seg_id) in existing:
            continue
        sentence = _call_ollama(_build_prompt(ctx, seg_id), "gemma3:12b-it-qat", 0.7)
        result = validate({"vod_id": vod_id, "rec_sentence": sentence}, ctx)
        if not result["pass"]:
            fails.append({
                "asset_nm": ctx["asset_nm"],
                "seg": seg_id,
                "sentence": sentence,
                "len": len(sentence) if sentence else 0,
                "reasons": result["fail_reasons"],
            })

print(f"실패 {len(fails)}건:\n")
for f in fails:
    print(f"[seg{f['seg']}] {f['asset_nm']}")
    print(f"  문장({f['len']}자): {f['sentence']}")
    print(f"  사유: {f['reasons']}\n")

## 7. 다음 단계

1. Google Drive에서 `colab_data/results.parquet` 다운로드
2. 로컬에서 DB 적재:
```bash
python gen_rec_sentence/scripts/ingest_results.py \
    gen_rec_sentence/data/colab_data/results.parquet
```
3. 사후검증 파이프라인 실행 (EXPLORATION_LOG.md §16 참조)

In [ ]:
# vLLM 설치 + 의존성
!pip install -q vllm aiohttp pandas pyarrow python-dotenv

# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, time, requests

# ── vLLM 서버 백그라운드 시작 ──
# AWQ INT4 양자화 모델: ~16GB VRAM → A100 40GB에서 여유 있게 동작
MODEL_ID = "hugging-quants/gemma-2-27b-it-AWQ-INT4"

proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_ID,
        "--dtype", "float16",
        "--quantization", "awq",
        "--max-model-len", "2048",        # rec_sentence 프롬프트는 짧으니 충분
        "--gpu-memory-utilization", "0.90",
        "--max-num-seqs", "16",           # 동시 16개 배치 처리
        "--port", "8000",
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"vLLM PID: {proc.pid}")
print("모델 로딩 중... (최초 실행 시 다운로드 포함 ~5분)")

# 서버 준비 대기 (최대 10분)
for i in range(120):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=3)
        if r.status_code == 200:
            print(f"\nvLLM 서버 준비 완료! ({(i+1)*5}초 소요)")
            print(r.json())
            break
    except requests.ConnectionError:
        if i % 12 == 0:
            print(f"  대기 중... ({(i+1)*5}초)")
else:
    print("⚠️  서버 시작 실패. 로그 확인:")
    !tail -20 /tmp/vllm.log

In [ ]:
import os

# ⚠️ Drive 폴더명 확인
PROJECT_ROOT = "/content/drive/MyDrive/dxschool"
OFFLINE_DIR = f"{PROJECT_ROOT}/gen_rec_sentence/data/colab_data"

assert os.path.exists(f"{OFFLINE_DIR}/vod_contexts.parquet"), \
    f"vod_contexts.parquet 없음! 로컬에서 export_for_colab.py 먼저 실행"

os.chdir(PROJECT_ROOT)
print(f"작업 디렉토리: {os.getcwd()}")

# 소규모 테스트 (10건, vLLM)
!python gen_rec_sentence/scripts/batch_generate.py \
    --offline gen_rec_sentence/data/colab_data \
    --backend vllm \
    --model hugging-quants/gemma-2-27b-it-AWQ-INT4 \
    --concurrency 16 \
    --limit 10 \
    --temperature 0.7

In [ ]:
# 전체 배치 (vLLM AWQ INT4, 동시 16개 — 예상 4~5시간)
# 200건마다 자동 저장, 세션 끊겨도 재실행 시 이어서 생성
!python gen_rec_sentence/scripts/batch_generate.py \
    --offline gen_rec_sentence/data/colab_data \
    --backend vllm \
    --model hugging-quants/gemma-2-27b-it-AWQ-INT4 \
    --concurrency 16 \
    --temperature 0.7